### Load the Basic Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2
import pickle

In [2]:
def detect_face(frame):
    detector=cv2.CascadeClassifier("haarcascade_frontalface_alt.xml")
    face=detector.detectMultiScale(frame,1.2)
    return face
def gray_scale(image):
    img=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
    return img
def cut_face(image,face_coords):
    faces=[]
    for (x,y,w,h) in face_coords:
        img=image[y:y+h,x:x+w]
        faces.append(img)
    return faces
def normalize_intensity(images):
    normalize_image=[]
    for image in images:
        img=cv2.equalizeHist(image)
        normalize_image.append(img)
    return normalize_image
def resize(images,size=(80,100)):
    resized_images=[]
    for img in images:
        image=cv2.resize(img,size)
        resized_images.append(image)
    return resized_images
def pipeline(image,face_coords):
    faces=cut_face(image,face_coords)
    faces=normalize_intensity(faces)
    faces=resize(faces)
    return faces
def plot_fxn(image,title=''):
    if image.shape==3:
        cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    plt.axis("off")
    plt.title(title)
    plt.imshow(image,cmap='gray')
    plt.show()
def draw_rectangle(image,coords):
    for (x,y,w,h) in coords:
        cv2.rectangle(image,(x,y),(x+w,y+h),(0,0,255),2)

In [3]:
def collect_data():
    images=[]
    labels=[]
    label_dic={}
    people=[person for person in os.listdir("DataSet1")]
    for i,person in enumerate(people):
        label_dic[i]=person
        for img in os.listdir("DataSet1/"+person):
            if img.endswith(".jpg"):
                imgs=cv2.imread("DataSet1/"+person+"/"+img,0)
                images.append(imgs)
                labels.append(i)
    return (images,labels,label_dic)

In [4]:
_,_,label_dic=collect_data()

In [5]:
label_dic

{0: 'naina', 1: 'unknown'}

#### Load Model

In [6]:
sc=pickle.load(open("scaling_model.pkl",'rb'))
pca=pickle.load(open("pca_model.pkl",'rb'))
model=pickle.load(open("face_recog_model.pkl",'rb'))

### GUI Face Recognition

In [8]:
cam=cv2.VideoCapture(1)
cv2.namedWindow("Face Recognition System !!!!")
while True:
    rect,frame=cam.read()
    gray=gray_scale(frame)
    face_coord=detect_face(gray)
    if len(face_coord)>0:
        faces=pipeline(gray,face_coord)
        for i,face in enumerate(faces):
            t=face.reshape(1,-1)
            tst=sc.transform(t)
            test=pca.transform(tst)

            pred=model.predict(test)
            name=label_dic[pred[0]]
            cv2.putText(frame,name,(face_coord[i][0],face_coord[i][1]-10),cv2.FONT_HERSHEY_PLAIN,3,(0,0,255))
            
    else:
        cv2.putText(frame,"No Face Found !!!",(5,60),cv2.FONT_HERSHEY_PLAIN,3,(0,0,255))
    cv2.putText(frame,"Enter 'Esc' to exit  ",(5,400),cv2.FONT_HERSHEY_PLAIN,3,(0,0,255))
    draw_rectangle(frame,face_coord)
    cv2.imshow("Face Recognition System !!!!",frame)
    if cv2.waitKey(5)==27:
        break
cam.release()
cv2.destroyAllWindows()

In [ ]:
frame=""

In [ ]:
gray=gray_scale(frame)
face_coord=detect_face(gray)
if len(face_coord)>0:
    faces=pipeline(gray,face_coord)
    for i,face in enumerate(faces):
        t=face.reshape(1,-1)
        tst=sc.transform(t)
        test=pca.transform(tst)

        pred=model.predict(test)
        name=label_dic[pred[0]]
        print(name)